# Land Type Classification — EuroSAT
### Digital Egypt Pioneers Initiative — AI & Data Science Track, Round 2 — Project 6

**Team members:** _(fill in names + assigned milestones)_

Uses the EuroSAT RGB dataset (Helber et al.) — 27,000 labeled Sentinel-2 image patches,
10 land use/land cover classes. No Copernicus account or raw satellite download needed.

Run in Colab with a GPU runtime for speed, or locally on CPU (slower but works).


## Setup

In [ ]:
!pip install -q torch torchvision scikit-learn matplotlib seaborn

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

sns.set_theme(style="whitegrid")
%matplotlib inline
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## Milestone 1: Data Collection, Exploration, and Preprocessing

### 1.1 Data Collection

EuroSAT is already labeled and geo-referenced, so "collection" here means downloading the
official release rather than querying a satellite catalog.


In [ ]:
# Downloads and extracts to data/raw/EuroSAT_RGB/<ClassName>/*.jpg
!python {ROOT}/src/download_eurosat.py

In [ ]:
from torchvision import datasets

DATA_DIR = ROOT / "data" / "raw" / "EuroSAT_RGB"
base = datasets.ImageFolder(str(DATA_DIR))
print("Classes:", base.classes)
print("Total images:", len(base))

### 1.2 Data Exploration

In [ ]:
# Class balance
counts = pd.Series(base.targets).map(dict(enumerate(base.classes))).value_counts()
plt.figure(figsize=(8, 4))
sns.barplot(x=counts.index, y=counts.values)
plt.xticks(rotation=45, ha="right")
plt.ylabel("Image count")
plt.title("EuroSAT class distribution")
plt.tight_layout()
plt.show()
counts

In [ ]:
# Sample grid — a few images per class
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for ax, cls in zip(axes.flat, base.classes):
    idx = base.targets.index(base.class_to_idx[cls])
    img, _ = base[idx]
    ax.imshow(img)
    ax.set_title(cls, fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Basic pixel statistics per channel (on a sample to keep this fast)
import random
random.seed(42)
sample_idx = random.sample(range(len(base)), 500)

means, stds = [], []
for i in sample_idx:
    img, _ = base[i]
    arr = np.array(img) / 255.0
    means.append(arr.mean(axis=(0, 1)))
    stds.append(arr.std(axis=(0, 1)))

means = np.array(means)
print("Per-channel mean (R,G,B):", means.mean(axis=0).round(3))
print("Per-channel std  (R,G,B):", np.array(stds).mean(axis=0).round(3))

### 1.3 Preprocessing and Feature Engineering

`src/dataset.py` handles this: a stratified 70/15/15 train/val/test split, normalization,
and augmentation (random horizontal/vertical flip, random 90° rotation) applied only to
the training split.


In [ ]:
from dataset import get_loaders

train_dl, val_dl, test_dl, classes = get_loaders(DATA_DIR, batch_size=64)
print("Classes:", classes)
print("Batches — train:", len(train_dl), "val:", len(val_dl), "test:", len(test_dl))

In [ ]:
# Visualize a few augmented training batches
xb, yb = next(iter(train_dl))
fig, axes = plt.subplots(1, 6, figsize=(15, 3))
for ax, img, label in zip(axes, xb[:6], yb[:6]):
    img = img.permute(1, 2, 0).numpy() * 0.5 + 0.5  # undo normalization for display
    ax.imshow(np.clip(img, 0, 1))
    ax.set_title(classes[label], fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Milestone 2: Advanced Data Analysis and Feature Engineering

### 2.1 Per-class mean brightness / channel analysis


In [ ]:
rows = []
for i in sample_idx:
    img, label = base[i]
    arr = np.array(img) / 255.0
    rows.append({
        "class": classes[label],
        "R_mean": arr[..., 0].mean(),
        "G_mean": arr[..., 1].mean(),
        "B_mean": arr[..., 2].mean(),
        "brightness": arr.mean(),
    })
df = pd.DataFrame(rows)
df.groupby("class")[["R_mean", "G_mean", "B_mean", "brightness"]].mean().round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.boxplot(data=df, x="class", y="brightness", ax=axes[0])
axes[0].tick_params(axis="x", rotation=45)
axes[0].set_title("Overall brightness by class")

sns.heatmap(df.drop(columns="class").corr(), annot=True, fmt=".2f", cmap="coolwarm", ax=axes[1])
axes[1].set_title("Channel correlation")
plt.tight_layout()
plt.show()

### 2.2 Dimensionality reduction (PCA) on channel-mean features

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

X = df[["R_mean", "G_mean", "B_mean"]].values
X_scaled = StandardScaler().fit_transform(X)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=df["class"], palette="tab10", s=25)
plt.title(f"PCA on RGB means (explained variance: {pca.explained_variance_ratio_.sum()*100:.1f}%)")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

### 2.3 Feature Engineering Summary

- Random horizontal/vertical flips + 90° rotations (`dataset.py`) — cheap augmentation that
  respects satellite imagery's lack of a "natural" orientation (unlike photos, top-down tiles
  have no canonical up).
- Per-channel normalization to `[-1, 1]` (mean/std = 0.5) for stable CNN training.
- No class rebalancing applied — EuroSAT classes are close to balanced already (see 1.2);
  add `WeightedRandomSampler` in `dataset.py` if you outsample a specific class later.


## Milestone 3: Model Development and Training

Two architectures, both defined in `src/model.py`:
- **`cnn`** — a small 3-block CNN trained from scratch
- **`resnet18`** — ImageNet-pretrained ResNet18, fine-tuned (transfer learning)


In [ ]:
from model import build_model

model = build_model("cnn", num_classes=len(classes)).to(device)
print(model)

### 3.1–3.4 Training, evaluation, confusion matrix

Handled by `src/train.py`, which does early stopping on validation loss and saves the
best checkpoint + a confusion matrix + loss/accuracy curves. Run it as a script:

In [ ]:
!python {ROOT}/src/train.py --model cnn --epochs 30
# !python {ROOT}/src/train.py --model resnet18 --epochs 15 --lr 0.0003

Or run the same loop inline here, if you want to inspect it step by step:

In [ ]:
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix

def run_epoch(model, loader, criterion, device, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * x.size(0)
            correct += (out.argmax(1) == y).sum().item()
            n += x.size(0)
    return total_loss / n, correct / n

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2)

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_loss, patience, wait = float("inf"), 5, 0

for epoch in range(1, 31):
    tl, ta = run_epoch(model, train_dl, criterion, device, optimizer)
    vl, va = run_epoch(model, val_dl, criterion, device)
    scheduler.step(vl)
    for k, v in zip(history, (tl, vl, ta, va)):
        history[k].append(v)
    print(f"epoch {epoch:02d}  train {tl:.4f}/{ta:.3f}  val {vl:.4f}/{va:.3f}")
    if vl < best_val_loss:
        best_val_loss, wait = vl, 0
        torch.save(model.state_dict(), ROOT / "models" / "best_cnn_inline.pt")
    else:
        wait += 1
        if wait >= patience:
            print("early stopping")
            break

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss"); axes[0].legend()
axes[1].plot(history["train_acc"], label="train")
axes[1].plot(history["val_acc"], label="val")
axes[1].set_title("Accuracy"); axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
def predict(model, loader, device):
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for x, y in loader:
            ps.append(model(x.to(device)).argmax(1).cpu().numpy())
            ys.append(y.numpy())
    return np.concatenate(ys), np.concatenate(ps)

model.load_state_dict(torch.load(ROOT / "models" / "best_cnn_inline.pt", map_location=device))
y_true, y_pred = predict(model, test_dl, device)
print(classification_report(y_true, y_pred, target_names=classes, digits=4))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=classes, yticklabels=classes)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

### 3.5 Hyperparameter tuning

Compare a few learning rates using validation accuracy.


In [ ]:
def hyperparam_search(lrs=(1e-3, 5e-4, 1e-4), epochs=10):
    results = []
    for lr in lrs:
        m = build_model("cnn", len(classes)).to(device)
        opt = torch.optim.Adam(m.parameters(), lr=lr)
        best_acc = 0
        for _ in range(epochs):
            run_epoch(m, train_dl, criterion, device, opt)
            _, va = run_epoch(m, val_dl, criterion, device)
            best_acc = max(best_acc, va)
        results.append({"lr": lr, "best_val_acc": best_acc})
    return pd.DataFrame(results).sort_values("best_val_acc", ascending=False)

# search_results = hyperparam_search()
# search_results

### 3.6 Compare against ResNet18 (transfer learning)

Run this as a script for a clean checkpoint: `python src/train.py --model resnet18 --epochs 15 --lr 0.0003`


## Next steps (Milestones 4 & 5)

- **Milestone 4**: `app/app.py` — Flask API that loads `models/best_<model>.pt` +
  `models/classes.json` and serves `/predict` on an uploaded image.
- **Milestone 5**: final report and presentation, in `docs/`.

Make sure `models/classes.json` and whichever checkpoint you keep are present before running
the API — `train.py` writes both automatically.
